# Physical Correctness & Input Verification Guide

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

MolScope treats **physical correctness and coordinate integrity** as foundational prerequisites. Downstream featurization, non-linear manifold learning, and machine learning models are only as valid as the underlying coordinates and topologies ingested into the pipeline.

### What You Will Learn
1. **Periodic Boundary Conditions (PBC) & Coordinate Wrapping**: Understanding how independent atomic wrapping tears covalent bonds and creates artificial 50–100 Å jumps.
2. **Automated Vectorized Detection**: Using `inspect_md_inputs` and `profile_system` to scan for broken covalent bonds and virtual $C\alpha_i - C\alpha_{i+1}$ peptide continuity.
3. **On-the-Fly Healing via `ImagingPolicy`**: Applying native unwrapping and solute centering without modifying raw files on disk.
4. **The Zero-Imputation Principle**: Why silently imputing missing molecular measurements with zeros or means catastrophically corrupts physical dynamics.
5. **Transparent Applicability Auditing**: How applicability contracts propagate through all pipeline matrices.


## 1. The Physical Problem: PBC Coordinate Wrapping

Molecular dynamics engines (OpenMM, GROMACS, AMBER, CHARMM) simulate macromolecules in periodic unit cells to eliminate boundary vacuum artifacts. During integration, coordinates are frequently "wrapped" into the primary unit cell `[0, L]` independently for each atom.

```text
       Wrapped (Broken Molecule)                     Imaged (Whole Molecule)
+-----------------------------------+     +-----------------------------------+
|  (C)                              |     |                                   |
|   | (broken bond across box edge) |     |            (CA)-(C)-(N)           |
|                                   |     |              |                    |
|                               (CA)|     |                                   |
+-----------------------------------+     +-----------------------------------+
      Raw distance: ~48.5 Å                        Raw distance: ~1.5 Å
      PBC distance: ~1.5 Å                         PBC distance: ~1.5 Å
```

When coordinates are wrapped without making macromolecules whole:
- **Covalent bonds break across unit cell faces**: One bonded atom sits at $x \approx 0.5$ Å while its partner wraps to $x \approx L - 0.5$ Å.
- **Catastrophic measurement distortion**: A $1.4$ Å covalent bond appears as a $50$ Å or $100$ Å separation in raw Cartesian coordinates.
- **Corrupted features**: Radius of gyration ($R_g$), RMSD, contact networks, and secondary structure metrics explode with unphysical jumps.


## 2. Inspecting Input Integrity with MolScope


In [ ]:
from pathlib import Path
from molscope.io.validation import inspect_md_inputs, InputIntegrityStatus
from molscope.profiling.system import profile_system

# Locate test fixture
pdb_path = Path("tests/fixtures/solvated_protein.pdb")
if not pdb_path.is_file():
    pdb_path = Path("../tests/fixtures/solvated_protein.pdb")

print(f"Target structure: {pdb_path}")

# Run comprehensive input inspection
report = inspect_md_inputs(
    topology=pdb_path,
    trajectory=pdb_path,
    frame_stride=1,
    check_pbc_wrapping=True,
    require_imaged=False,
)

print(f"Input Integrity Status: {report.status}")
if report.warnings:
    print(f"Detected {len(report.warnings)} warnings:")
    for w in report.warnings[:5]:
        print(f"  - {w}")
else:
    print("Coordinates are whole, finite, and properly imaged.")


## 3. System Profiling and Capability Assessment


In [ ]:
# Profile the system composition and molecular classes
profile = profile_system(pdb_path, pdb_path)

print(f"System ID:            {profile.system_id}")
print(f"Atom count:           {profile.atom_count}")
print(f"Residue count:        {profile.residue_count}")
print(f"Has protein:          {profile.has_protein}")
print(f"Has solvent:          {profile.has_solvent}")
print(f"Has nucleic acids:    {profile.has_nucleic_acid}")
print(f"Has ligands:          {profile.has_ligand}")
print(f"Wrapped coordinates:  {profile.wrapped_coordinates_detected}")


## 4. On-The-Fly Coordinate Healing via `ImagingPolicy`

MolScope allows on-the-fly unwrapping and solute centering during trajectory iteration.
This guarantees that feature extraction operates on physically whole molecules without requiring permanent file copies.


In [ ]:
from molscope.io.transformations import ImagingPolicy, apply_imaging_transformations
import MDAnalysis as mda

u = mda.Universe(str(pdb_path))

# Configure on-the-fly imaging policy:
# 1. unwrap: bonds across box faces are stitched together
# 2. center_selection: centers protein solute in box
# 3. wrap: solvent molecules remain compactly inside box
policy = ImagingPolicy(
    unwrap=True,
    center_selection="protein",
    center_to="box",
    wrap=False,
)

applied = apply_imaging_transformations(u, policy)
print(f"Successfully applied transformations: {applied}")


## 5. The Zero-Imputation Principle

In machine learning and statistics, missing data is often imputed using column means, medians, or zeros. **In physical chemistry, silent imputation is catastrophic**:
- Setting a missing distance or contact feature to `0.0` asserts that atoms are overlapping or forming intense physical interactions when no measurement existed.
- Filling mean values across heterogeneous conditions obscures real conformational transitions and creates synthetic data points that violate physical laws.

MolScope enforces a strict **Zero-Imputation Policy**:
> Missing, non-finite, or inapplicable features must propagate transparently with explicit statuses and human-readable audit reasons throughout all matrices, quality reports, and final rankings.

| Pipeline Stage | Output Artifact | Propagation Mechanism |
| :--- | :--- | :--- |
| **Applicability Contract** | `FeatureAssessment` | Flags status (`APPLICABLE`, `NOT_APPLICABLE`, `MISSING_INPUT`, `UNSUPPORTED`) before values are computed. |
| **Feature Extraction** | `extraction_evidence.json` | Records all assessments and missing capabilities per trajectory replica. |
| **Quality Assessment** | `feature_quality_metrics.csv` | Explicit `Eligibility_Status` (`eligible`, `unavailable`, `insufficient_valid_frames`) and `Ineligibility_Reason`. |
| **SAM-MFP Prioritization** | `pareto_feature_candidates.csv` | Explicit `SAM_MFP_Status` (`pareto_candidate`, `dominated`, `ineligible`) and `SAM_MFP_Reason`. |
| **Predictive ML LOFO** | `predictive_feature_ranking.csv` | Non-finite features excluded from training without imputation; appended with `Ranking_Status = "inapplicable"`. |


In [ ]:
from molscope.contracts import FeatureAssessment, FeatureStatus

assessment = FeatureAssessment(
    feature_name="protein.salt_bridges",
    status=FeatureStatus.APPLICABLE,
    reason="All standard acidic and basic residues present in topology.",
)

print(f"Feature: {assessment.feature_name}")
print(f"Status:  {assessment.status.name}")
print(f"Reason:  {assessment.reason}")
